# 🌐 RSNA Knee Multimodal AI - Phase 5: Vision-Language Alignment & Distillation

This notebook demonstrates multimodal contrastive learning (MedCLIP-style) between **3D Multi-Sequence Knee MRI Volumes** and **Multilingual Radiology Reports**.

### Key Concepts:
1. **ClinicalTextEncoder**: Encodes unstructured multilingual clinical texts into fixed-dimension dense representations.
2. **MultimodalContrastiveModel**: Projects vision and text representations onto a shared hypersphere using symmetric InfoNCE loss.
3. **Cross-Modal Similarity Heatmap**: Computes pairwise cosine similarities across studies, aligning visual abnormalities with clinical findings.
4. **MultimodalFusionClassifier**: Fuses MRI visual embeddings with report text for joint dual-modality classification.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from src.models import MultiViewKneeModel, TARGET_COLS
from src.multimodal import (
    ClinicalTextEncoder, 
    MultimodalContrastiveModel, 
    MultimodalFusionClassifier
)

sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. 📝 Multilingual Clinical Text Encoding
Free-text reports in Spanish, French, Dutch, and English contain nuanced findings. Let's encode sample reports.

In [ ]:
text_encoder = ClinicalTextEncoder(vocab_size=10000, out_dim=256).to(device)

sample_reports = [
    "Rotura completa del ligamento cruzado anterior con derrame articular moderado.", # Study 1 (Spanish ACL)
    "Meniscopathie mediale degenerative avec lesion complexe de la corne posterieure.", # Study 2 (French Meniscus)
    "Gonartrose femorotibiaal met kraakbeendefect en osteofyten.",                    # Study 3 (Dutch OA)
    "Severe bone marrow contusion of the lateral femoral condyle and tibia plateau."    # Study 4 (English Contusion)
]

text_embeddings = text_encoder(sample_reports, device=device)
print(f"Encoded {len(sample_reports)} reports -> Dense Embedding Shape: {text_embeddings.shape}")

## 2. 🩻 Vision-Language Contrastive Hypersphere Alignment
Let's project both 3D MRI volume embeddings and report embeddings into the shared 256-dimensional semantic space and compute the cross-modal similarity matrix.

In [ ]:
vision_model = MultiViewKneeModel(num_classes=12, feature_dim=128, fused_dim=256).to(device)
contrastive_model = MultimodalContrastiveModel(
    vision_model=vision_model,
    text_encoder=text_encoder,
    latent_dim=256
).to(device)

# Simulate 4 multi-view MRI study volumes: [B=4, 3, Depth=16, H=224, W=224]
mock_volumes = torch.randn(4, 3, 16, 224, 224).to(device)

z_v, z_t, sim_matrix = contrastive_model(mock_volumes, sample_reports)
cosine_sims = torch.matmul(z_v, z_t.t()).detach().cpu().numpy()

# Visualize cross-modal similarity matrix
fig, ax = plt.subplots(figsize=(8, 6))
labels = ['Study 1 (ACL)', 'Study 2 (Meniscus)', 'Study 3 (OA)', 'Study 4 (Contusion)']
sns.heatmap(cosine_sims, annot=True, fmt='.2f', cmap='viridis', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Cross-Modal Vision-Language Cosine Similarity Matrix', fontsize=12, fontweight='bold')
ax.set_xlabel('Clinical Report Embeddings', fontsize=10)
ax.set_ylabel('MRI Volume Visual Embeddings', fontsize=10)
plt.tight_layout()
plt.show()

## 3. 🤝 Dual-Modality Joint Fusion Inference
During deployment, studies with both MRI images and reports can leverage `MultimodalFusionClassifier` for maximum diagnostic precision.

In [ ]:
fusion_classifier = MultimodalFusionClassifier(
    contrastive_model=contrastive_model,
    num_classes=12,
    fused_dim=256
).to(device)

joint_logits = fusion_classifier(mock_volumes, sample_reports)
joint_probs = torch.sigmoid(joint_logits).detach().cpu().numpy()

pred_df = pd.DataFrame(joint_probs, columns=TARGET_COLS, index=labels)
print("=== Multimodal Joint Pathology Predictions ===")
display(pred_df.round(3))